In [114]:
from sklearn.model_selection import train_test_split
from shutil import move , copy2
import yaml


In [109]:
train_path = data_path / "Train"
test_path = data_path / "Test"

for class_name in classes:
    images_path = train_path / class_name / "images"
    labels_path = train_path / class_name / "labels"

    test_images_path = test_path / class_name / "images"
    test_labels_path = test_path / class_name / "labels"

    test_images_path.mkdir(parents=True, exist_ok=True)
    test_labels_path.mkdir(parents=True, exist_ok=True)

    image_file = [
        image
        for image in images_path.iterdir()
        if image.suffix.lower() in image_extensions
        and (labels_path / f"{image.stem}.txt").exists()
    ]
    print(len(image_file))

    train_images, test_images = train_test_split(
        image_file, test_size=0.2, random_state=42, shuffle=True
    )
    print(len(train_images) , len(test_images))
    

    for image in test_images:
        label = labels_path / f"{image.stem}.txt"

        move(str(image), str(test_images_path / image.name))
        move(str(label), str(test_labels_path / label.name))

    print(f"{class_name}: Train = {len(train_images)}, Test = {len(test_images)}")

    


1153
922 231
Glioma: Train = 922, Test = 231
1449
1159 290
Meningioma: Train = 1159, Test = 290
711
568 143
No Tumor: Train = 568, Test = 143
1424
1139 285
Pituitary: Train = 1139, Test = 285


In [110]:
for split in ["Train", "Val", "Test"]:
    print(f"\n{split}")
    for class_name in classes:
        class_path = data_path / split / class_name

        images_path = class_path / "images"
        labels_path = class_path / "labels"

        image_count = sum(
            file.suffix.lower() in image_extensions for file in images_path.iterdir()
        )

        label_count = len(list(labels_path.glob("*.txt")))

        print(f"{class_name}: {image_count} images | {label_count} labels")



Train
Glioma: 922 images | 922 labels
Meningioma: 1159 images | 1159 labels
No Tumor: 568 images | 568 labels
Pituitary: 1139 images | 1139 labels

Val
Glioma: 136 images | 136 labels
Meningioma: 140 images | 140 labels
No Tumor: 98 images | 98 labels
Pituitary: 136 images | 136 labels

Test
Glioma: 231 images | 231 labels
Meningioma: 290 images | 290 labels
No Tumor: 143 images | 143 labels
Pituitary: 285 images | 285 labels


In [112]:
data_config = {
    "path": str(data_path.resolve()),
    "train": "Train",
    "val": "Val",
    "test": "Test",
    "nc": 4,
    "names": {
        0: "Glioma",
        1: "Meningioma",
        2: "No Tumor",
        3: "Pituitary",
    },
}

yaml_path = data_path / "data.yaml"

with open(yaml_path, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        data_config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

In [115]:
output_path = Path.cwd() / "data" / "processed" / "brain_mri_yolo"

splits = {
    "Train": "train",
    "Val": "val",
    "Test": "test",
}

for source_split, target_split in splits.items():
    for class_name in classes:
        images_path = data_path / source_split / class_name / "images"
        labels_path = data_path / source_split / class_name / "labels"

        for image in images_path.iterdir():
            if image.suffix.lower() not in image_extensions:
                continue

            label = labels_path / f"{image.stem}.txt"

            if not label.exists():
                continue

            new_name = f"{class_name.lower().replace('', '_')}_{image.stem}"

            target_iamge = (
                output_path
                / "images"
                / target_split
                / f"{new_name}{image.suffix.lower()}"
            )

            target_label = (output_path / "labels" / target_split / f"{new_name}.txt")

            target_iamge.mkdir(parents=True, exist_ok=True)
            target_label.mkdir(parents=True, exist_ok=True)

            copy2(image, target_iamge)
            copy2(label, target_label)

In [ ]:
for split in ["train", "val", "test"]:
    images_path = output_path / "images" / split
    labels_path = output_path / "labels" / split

    images = [
        file
        for file in images_path.iterdir()
        if file.suffix.lower() in image_extensions
    ]

    labels = list(labels_path.glob("*.txt"))

    print(f"{split}: {len(images)} images | {len(labels)} labels")

FileNotFoundError: [Errno 2] No such file or directory: '/data/processed/brain_mri_yolo/train/images'